# Experimental RAG Chatbot with Input Documents

This notebook explores how to build a retrieval-augmented generation (RAG) chatbot over a technical PDF. The pipeline extracts structured text and images, converts document content into searchable chunks, embeds those chunks into vectors, retrieves relevant context with FAISS, and asks an open-source language model to answer user questions.

The goal is experimentation and learning rather than production deployment. The retrieved context is intended to ground answers and reduce unsupported responses, while the prompt explicitly instructs the model to say when it does not know an answer.

## 1. Set Up the RAG Experiment

Install the libraries used for document parsing, embedding generation, vector search, prompt orchestration, and local language-model inference. The notebook combines LangChain components with FAISS, a sentence-transformer embedding model, and a GGUF instruction model loaded through `llama-cpp-python`.

In [ ]:
!pip install langchain==0.2.5 faiss-cpu cohere==5.5.8 langchain-community==0.2.5 rank_bm25==0.2.2 sentence-transformers==3.0.1 transformers
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.78


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.6/974.6 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 35.9 MB/s eta 0:00:00


## 2. Load the Open-Source Generation Model

Download and initialize the Phi-3 Mini GGUF model. This model serves as the generation component of the RAG system: it receives the retrieved document context and the user's question, then produces the final response. The context window and token limit control how much information can be supplied at inference time.

In [ ]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2025-04-27 12:00:05--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.97, 13.35.202.121, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.97|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/41/c8/41c860f65b01de5dc4c68b00d84cead799d3e7c48e38ee749f4c6057776e2e9e/5d99003e395775659b0dde3f941d88ff378b2837a8dc3a2ea94222ab1420fad3?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1745758805&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0NTc1ODgwNX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzQxL2M4LzQxYzg2MGY2NWIwMWRlNWRjNGM2OGIwMGQ4NGNlYWQ3OTlkM2U3YzQ4ZTM4ZWU3NDlmNGM2MDU3Nzc2ZTJlOWUvNWQ5OTAwM2UzOTU3NzU2NTliMGRkZTNmOTQxZDg4

In [ ]:
# Importing embedding model from Hugging face {OPEN SOURCE}
import torch
# Import a GGUF format model and see how much time it takes

from langchain import LlamaCpp

# Make sure the model path is correct for your system
llm = LlamaCpp(
    model_path = "/content/Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers= -1,
    max_tokens = 500,
    n_ctx = 2048,
    seed = 42,
    verbose = False
)

## 3. Parse the Input PDF into Structured Content

Extract text and embedded images from the service manual while preserving a hierarchy based on delivery type, section, and subsection. Font size, character flags, and text position are used as heuristics to identify headings, content, and caution or danger notes. The result is a nested dictionary that keeps document structure available for later retrieval.

In [ ]:
import fitz  # PyMuPDF
import os
import json

def extract_text_and_images_with_metadata(pdf_path, output_dir="output_images"):

  os.makedirs(output_dir, exist_ok=True)
  doc = fitz.open(pdf_path)
  structured_data = {}

  current_delivery_type = None
  current_section = None
  current_subsection = None

  for page_num in range(len(doc)):

    page = doc.load_page(page_num)
    blocks = page.get_text("dict")["blocks"]

    for block in blocks:
      if 'lines' in block:
        for line in block["lines"]:
          for span in line["spans"]:
              text = span["text"].strip()
              font_size = span["size"]
              font_name = span["font"]
              char_flags = span["char_flags"]
              origin = span["origin"]

              if not text:
                  continue

              # Identify "Delivery Type, Section & Subsections" in document based on Font Size
              if font_size > 17 and font_size < 22 and char_flags == 16:  # Adjust this threshold as needed
                  current_delivery_type = text
                  structured_data[current_delivery_type] = {}

              elif font_size > 14 and font_size < 17 and current_delivery_type  and char_flags == 16:
                  current_section = text
                  structured_data[current_delivery_type][current_section] = {"content": "","caution or danger": "", "images": []}
                  current_subsection = None

              elif font_size > 13 and font_size < 15 and current_section and char_flags == 16:
                  current_subsection = text
                  structured_data[current_delivery_type][current_section][current_subsection] = {"content": "","caution or danger": "", "images": []}

              elif font_size > 12 and font_size < 13 and current_subsection and char_flags == 16 and text == "CAUTION":
                  structured_data[current_delivery_type][current_section][current_subsection]["caution or danger"] += text + ": "

              elif origin[0] > 74 and origin[0] < 80 and current_subsection and char_flags == 16 :
                  structured_data[current_delivery_type][current_section][current_subsection]["caution or danger"] += text + " "

              elif origin[0] > 74 and origin[0] < 80 and char_flags == 16 :
                  structured_data[current_delivery_type][current_section]["caution or danger"] += text + " "

              elif current_section and char_flags == 16 and origin[1] < 700:  # Store text under "content" section of "current_subsection"
                  if current_subsection:
                    structured_data[current_delivery_type][current_section][current_subsection]["content"] += text + " "
                  else:
                    structured_data[current_delivery_type][current_section]["content"] += text + " "

      # Condition to extract the respective image from the the document
      else:
        typ = block['type']
        if typ == 1:
          image_bytes = block["image"]
          image_extract_type = block["ext"]
          image_filename = os.path.join(output_dir, f"image_{current_subsection}_{page_num + 1}.{image_extract_type}")

          with open(image_filename, "wb") as img_file:
            img_file.write(image_bytes)

          # Attach images to the current section or subsection
          if current_subsection:
            structured_data[current_delivery_type][current_section][current_subsection]["images"].append(image_filename)
          else:
            structured_data[current_delivery_type][current_section]["images"].append(image_filename)

  return structured_data

# Usage
pdf_path = "volvo-trucks-basic-service-manual-pages.pdf"
Extracted_Text = extract_text_and_images_with_metadata(pdf_path)
print("Data extracted and saved successfully!")

Data extracted and saved successfully!


In [ ]:
Extracted_Text

{'Design and Function': {'Clutch': {'content': '',
   'caution or danger': '',
   'images': [],
   'General': {'content': 'For further information concerning component speciﬁca- tions see service information in Group 1, “Oil and Filter Change Intervals for Volvo Components,” publication number 175–001, and appropriate vendor literature. Hydraulic ﬂuid in the clutch system collects moisture from the air and will eventually hold enough moisture to affect the metal surfaces in the system unless removed. Replace the ﬂuid at the recommended intervals or more frequently. The clutch pedal play (1) is given by the clearance be- tween the plunger and the piston (1a) in the master cylinder. Thus the pedal will always have a play, regard- less of the clutch adjustment. The correct play is adjusted with the upper adjusting screw (1b) in the pedal carrier. Fig. 1: Clutch ',
    'caution or danger': '',
    'images': ['output_images/image_General_1.png']},
   'Volvo Clutch Slave Cylinder': {'content

## 4. Convert Structured Data into Retrieval Chunks

Flatten the nested document dictionary into readable text records. Each chunk includes its hierarchy path, such as delivery type, section, and subsection, so retrieved passages retain useful source context instead of becoming isolated sentences.

In [ ]:
# Defining a function to flatten the extracted text
def flatten_dict(d, parent_key='', sep=' > '):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep))
        else:
            items.append(f"{new_key}: {v}")
    return items
chunks = flatten_dict(Extracted_Text)
chunks

['Design and Function > Clutch > content: ',
 'Design and Function > Clutch > caution or danger: ',
 'Design and Function > Clutch > images: []',
 'Design and Function > Clutch > General > content: For further information concerning component speciﬁca- tions see service information in Group 1, “Oil and Filter Change Intervals for Volvo Components,” publication number 175–001, and appropriate vendor literature. Hydraulic ﬂuid in the clutch system collects moisture from the air and will eventually hold enough moisture to affect the metal surfaces in the system unless removed. Replace the ﬂuid at the recommended intervals or more frequently. The clutch pedal play (1) is given by the clearance be- tween the plunger and the piston (1a) in the master cylinder. Thus the pedal will always have a play, regard- less of the clutch adjustment. The correct play is adjusted with the upper adjusting screw (1b) in the pedal carrier. Fig. 1: Clutch ',
 'Design and Function > Clutch > General > caution 

In [ ]:
# Importing embedding model from Hugging face {OPEN SOURCE}
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embedded_vectors = embedding_model.embed_documents(chunks)

<ipython-input-4-b52f9965b572>:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.

## 5. Generate Embeddings and Build the Vector Database

Use the `all-MiniLM-L6-v2` sentence-transformer to map each text chunk into a dense vector representation. FAISS stores those vectors and supports nearest-neighbor search, allowing the system to identify chunks that are semantically related to a user's question.

In [ ]:
# Creating vector Db using FAISS

from langchain.vectorstores import FAISS

vector_store = FAISS.from_texts(chunks, embedding_model)


## 6. Create the Retrieval-Augmented Generation Chain

Define a prompt that combines retrieved context with the user's question. The `RetrievalQA` chain queries the FAISS-backed retriever, inserts the relevant passages into the prompt, and sends the grounded request to the local language model. The instruction to avoid inventing answers is a basic guardrail for this experiment.

In [ ]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""<|user|>
Context:
{context}

Provide a concise answer to the following question using relevant information provided above under Context:
If you don't know the answer, just say "Sorry, I don't know" — don't make it up.
{question}<|end|>
<|assistant|>
"""
)

RAG = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type = 'stuff',
    retriever = vector_store.as_retriever(),
    chain_type_kwargs = {
        "prompt":prompt_template
    },
    verbose = True
)


In [ ]:
RAG.invoke("Which clutch do we have apart from Volvo clutch?")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Which clutch do we have apart from Volvo clutch?',
 'result': ' Apart from the Volvo clutch, there are other types of clutches that may feature a wear indicator for the slave cylinder, as mentioned in the context under "Other Clutch Slave Cylinder." These alternative clutches include non-Volvo models, which can have similar components like mounting bolts, clutch clevis pin, and connections to the master cylinder. However, specific designs or manufacturers are not detailed in the provided information.'}

### 6.1 Ask Questions over the Indexed Manual

The query cells test the chatbot with questions about clutches, PTO servicing, driveshaft grease, driveshaft cautions, and air dryers. These examples probe whether semantic retrieval finds the relevant manual sections and whether the generation model uses that context in its response.

In [ ]:
RAG.invoke("What about the PTO, do we have any caution while taking care of it during servicing?")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What about the PTO, do we have any caution while taking care of it during servicing?',
 'result': " During servicing, a caution is advised if the Power Take-Off (PTO) operates continuously for over 15 minutes at a time or has a continuous power output exceeding 55 kW (75 hp). It's important to install a transmission oil heat exchanger under these conditions to prevent oil overheating and potential transmission damage."}

In [ ]:
RAG.invoke("What kind of grease I should use when it comes to driveshaft?")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What kind of grease I should use when it comes to driveshaft?',
 'result': ' You should use a lithium-based grease with EP additives and have a consistency of NLGI No. 2 for lubricating the driveshaft U-joints. Do not use conventional chassis grease, and ensure that new grease flushes out abrasive contaminants from each bearing properly.'}

In [ ]:
RAG.invoke("Any kind of caution should I take while working with driveshaft?")['result']



> Entering new RetrievalQA chain...

> Finished chain.


' When working with a driveshaft, ensure to correctly grease the U-joints or slip-joints to prevent component failure that could lead to separation from the vehicle, causing potential major damage, loss of control, and serious personal injury or death.'

In [ ]:
RAG.invoke("how often I should take care of cartridge in ari dryers?")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'how often I should take care of cartridge in ari dryers?',
 'result': ' Dryer manufacturers recommend changing the cartridge every 2 to 3 years, but intervals may need adjustment based on the specific application. Also, if there is more water being drained than usual during daily tank emptying, consider replacing the cartridge sooner as an indicator for maintenance needs. The only air dryer with a regular annual maintenance routine is the Midland model, which requires changing its coalescent filter every year.'}